# Multi-Asset Strategic Allocation, Rebalancing & Trend-Following Overlay Framework

## Universe

Equities (US, Developed Markets ex-US, Emerging Markets), Crypto (Bitcoin), Bonds, Diversifers (Private Markets, Real Estate, Commodities, Precious Metals, Alternative Strategies), Cash.

## Summary

This framework converts a long-run multi-asset strategic allocation into actionable portfolio management instructions while preserving a clear distinction between **strategic allocation** and **tactical trend-following exposure**.

It is designed to be used in tandem with a separate trend-following model. That model independently generates one of three regime signals for each asset class:

- **ACCUMULATION** — the asset class is eligible for staged entry or continued accumulation toward its full strategic target (either: a) early Bullish reversal at the end of a Bear market, or b) temporary pullback during a wider Bull run).
- **HOLD** — the strategic allocation remains active, but no new purchases are generated by the intra-month accumulation engine. The asset class is eligible for profit-taking during the monthly rebalancing.
- **EXIT** — the strategic allocation itself is preserved for long-run planning purposes, but the active target exposure is temporarily reduced to zero until the trend-following model generates a new re-entry / accumulation signal.

The framework therefore separates three layers:

1. **Strategic allocation** — the long-run GBP and percentage allocation assigned to each asset class.
2. **Trend-following overlay** — determines whether that strategic allocation is currently active, being accumulated, or temporarily excluded.
3. **Execution engine** — compares current exposure with active target exposure and generates either monthly sell-only rebalancing instructions or intra-month buy-only DCA (Dollar-Cost Averaging) instructions.

Settings:

a) **Monthly rebalance** — sell-only. Exit positions and automatically trim overweighted exposures.

b) **Intra-month accumulation** — buy-only. Controlled entry or further accumulation of postions using the DCA approach spread over the month.

## Notes

A trend-following **EXIT** does **not** permanently reallocate the asset class's strategic capital to other assets. Instead, the corresponding strategic exposure is reported as **Trend-Excluded Exposure**. When the signal later changes to **ACCUMULATION**, the asset class's full strategic target becomes active again and the framework can calculate the amount required to rebuild the position.


In [ ]:
# ============================================================
# STRATEGIC ASSET ALLOCATION & TREND-FOLLOWING OVERLAY FRAMEWORK
# ============================================================
#
# Designed for Google Colab / JupyterLab
#
# This notebook should be used in tandem with a SEPARATE
# trend-following model. The external trend model generates
# one of three signals for each asset class:
#
#       ACCUMULATION
#       HOLD
#       EXIT
#
# These signals are entered manually in BLOCK 1 below.
#
# CORE LOGIC
# ----------
# 1. Define cash capital and desired portfolio exposure.
# 2. Enter long-run strategic GBP targets for each asset class.
# 3. Enter the current trend-following signal for each asset.
# 4. Enter current GBP exposures.
# 5. Select the execution mode:
#
#       MONTHLY_REBALANCE
#       INTRAMONTH_ACCUMULATION
#
# STRATEGIC VS ACTIVE TARGETS
# ---------------------------
# Strategic GBP:
#   The long-run allocation. This does NOT change merely
#   because the trend-following model generates EXIT.
#
# Active Target GBP:
#   The exposure currently authorised by the trend overlay.
#
#   ACCUMULATION -> Active target = Strategic target
#   HOLD         -> Active target = Strategic target
#   EXIT         -> Active target = £0
#
# An EXIT therefore creates "Trend-Excluded Exposure" rather
# than permanently changing the strategic asset allocation.
#
# MONTHLY_REBALANCE
# -----------------
# - SELL ONLY.
# - If current exposure exceeds the upper active-target band,
#   sell back to the upper band.
# - Underweights are NOT purchased.
# - EXIT positions have an active target of £0 and are therefore
#   sold when MONTHLY_REBALANCE is run.
#
# INTRAMONTH_ACCUMULATION
# -----------------------
# - BUY ONLY.
# - Only assets marked ACCUMULATION can be purchased.
# - Buys are based on the gap between current exposure and
#   active target exposure.
# - The purchase amount can be split into DCA instalments.
# - EXIT sales are not executed in this mode; they are flagged
#   for the next MONTHLY_REBALANCE run.
#
# ============================================================


# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    display = print


In [ ]:
# ============================================================
# BLOCK 1 — STRATEGIC ALLOCATION & TREND SIGNAL INPUTS
# ============================================================
#
# THIS IS ONE OF THE TWO MAIN BLOCKS YOU EDIT.
#
# CASH_CAPITAL_GBP:
#   Actual capital/equity available to support the portfolio.
#
# EXPOSURE_MULTIPLIER:
#   1.00 = 100% strategic exposure
#   1.20 = 120% strategic exposure
#   0.80 = 80% strategic exposure
#
# NOTIONAL_PORTFOLIO_GBP:
#   Calculated automatically as:
#
#       cash capital × exposure multiplier
#
# Strategic targets should normally sum to this value.
#
# ------------------------------------------------------------

CASH_CAPITAL_GBP = 100_000

EXPOSURE_MULTIPLIER = 1.20

NOTIONAL_PORTFOLIO_GBP = CASH_CAPITAL_GBP * EXPOSURE_MULTIPLIER


# ------------------------------------------------------------
# EXECUTION MODE
# ------------------------------------------------------------
#
# Choose ONE:
#
# "MONTHLY_REBALANCE"
# "INTRAMONTH_ACCUMULATION"
#
# ------------------------------------------------------------

RUN_MODE = "INTRAMONTH_ACCUMULATION"


# ------------------------------------------------------------
# REBALANCING BANDS
# ------------------------------------------------------------
#
# These are RELATIVE to each asset's ACTIVE target.
#
# Example:
#
# Strategic target weight = 20%
# Trend signal            = HOLD
# Active target weight    = 20%
# Upper band              = 10%
#
# Upper threshold =
# 20% × 1.10 = 22%
#
# Lower threshold =
# 20% × 0.90 = 18%
#
# If the same asset receives an EXIT signal, its active target
# and both bands become zero while its strategic target remains
# unchanged in the strategic allocation table.
#
# ------------------------------------------------------------

UPPER_BAND_PCT = 0.10
LOWER_BAND_PCT = 0.10


# ------------------------------------------------------------
# ACCUMULATION / DCA SETTINGS
# ------------------------------------------------------------
#
# ACCUMULATION_FILL_FRACTION:
#
# 1.00 = close 100% of the active-target shortfall this month
# 0.50 = close 50% of the active-target shortfall this month
# 0.25 = close 25% of the active-target shortfall this month
#
# DCA_INSTALLMENTS:
#
# Number of equal purchases through which the recommended
# monthly accumulation amount will be divided.
#
# ------------------------------------------------------------

ACCUMULATION_FILL_FRACTION = 1.00

DCA_INSTALLMENTS = 4


# ------------------------------------------------------------
# STRATEGIC ASSET UNIVERSE + TREND-FOLLOWING SIGNALS
# ------------------------------------------------------------
#
# Enter:
#
#       strategic_gbp
#       status
#
# Allowed statuses:
#
#       "ACCUMULATION"
#       "HOLD"
#       "EXIT"
#
# IMPORTANT:
#
# strategic_gbp is the LONG-RUN strategic allocation and is not
# changed by the trend-following signal.
#
# The status should be taken from the separate trend-following
# model:
#
# ACCUMULATION
#   Strategic target remains active and the asset becomes
#   eligible for staged buying in INTRAMONTH_ACCUMULATION mode.
#
# HOLD
#   Strategic target remains active. No intra-month buying is
#   generated merely because the asset is below target.
#
# EXIT
#   Strategic target is preserved for long-run planning, but
#   ACTIVE target exposure becomes £0 until a future re-entry
#   signal is generated.
#
# An EXIT allocation is NOT redistributed automatically to
# other asset classes.
#
# ------------------------------------------------------------

STRATEGIC_ALLOCATION = {

    "EQUITIES_US": {
        "strategic_gbp": 48_000,
        "status": "HOLD",
    },

    "EQUITIES_DM_EX_US": {
        "strategic_gbp": 15_000,
        "status": "HOLD",
    },

    "EQUITIES_EM": {
        "strategic_gbp": 15_000,
        "status": "EXIT",
    },

    "BITCOIN": {
        "strategic_gbp": 12_000,
        "status": "ACCUMULATION",
    },

    "BONDS": {
        "strategic_gbp": 12_000,
        "status": "HOLD",
    },

    "DIVERSIFIERS": {
        "strategic_gbp": 12_000,
        "status": "HOLD",
    },

    "CASH": {
        "strategic_gbp": 6_000,
        "status": "HOLD",
    },
}


In [ ]:
# ============================================================
# BLOCK 2 — CURRENT PORTFOLIO
# ============================================================
#
# THIS IS THE OTHER MAIN BLOCK YOU EDIT.
#
# Enter the CURRENT GBP EXPOSURE of each asset class.
#
# These are exposure / notional values, not necessarily the
# amount of cash or margin committed to the position.
#
# Every asset in STRATEGIC_ALLOCATION should appear here.
#
# ------------------------------------------------------------

CURRENT_PORTFOLIO_GBP = {

    "EQUITIES_US": 54_000,
    "EQUITIES_DM_EX_US": 16_000,
    "EQUITIES_EM": 11_000,
    "BITCOIN": 9_000,
    "BONDS": 10_000,
    "DIVERSIFIERS": 15_000,
    "CASH": 5_000,

}


# ============================================================
# VALIDATION
# ============================================================

VALID_STATUSES = {
    "ACCUMULATION",
    "HOLD",
    "EXIT",
}

VALID_RUN_MODES = {
    "MONTHLY_REBALANCE",
    "INTRAMONTH_ACCUMULATION",
}


if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(
        f"RUN_MODE must be one of {VALID_RUN_MODES}"
    )


if CASH_CAPITAL_GBP <= 0:
    raise ValueError("CASH_CAPITAL_GBP must be greater than zero.")


if EXPOSURE_MULTIPLIER < 0:
    raise ValueError("EXPOSURE_MULTIPLIER cannot be negative.")


if DCA_INSTALLMENTS < 1:
    raise ValueError("DCA_INSTALLMENTS must be at least 1.")


if not 0 <= ACCUMULATION_FILL_FRACTION <= 1:
    raise ValueError(
        "ACCUMULATION_FILL_FRACTION must be between 0 and 1."
    )


for asset, data in STRATEGIC_ALLOCATION.items():

    data["status"] = data["status"].upper()

    if data["status"] not in VALID_STATUSES:
        raise ValueError(
            f"{asset}: invalid status '{data['status']}'. "
            f"Use one of {VALID_STATUSES}"
        )

    if data["strategic_gbp"] < 0:
        raise ValueError(
            f"{asset}: strategic_gbp cannot be negative."
        )


# Check that both dictionaries contain the same assets.

strategic_assets = set(STRATEGIC_ALLOCATION.keys())
current_assets = set(CURRENT_PORTFOLIO_GBP.keys())

missing_current = strategic_assets - current_assets
extra_current = current_assets - strategic_assets

if missing_current:
    raise ValueError(
        f"Missing current portfolio values for: {missing_current}"
    )

if extra_current:
    raise ValueError(
        f"Current portfolio contains unknown assets: {extra_current}"
    )


# ============================================================
# BUILD STRATEGIC + ACTIVE ALLOCATION TABLE
# ============================================================

rows = []

for asset, data in STRATEGIC_ALLOCATION.items():

    strategic_gbp = float(data["strategic_gbp"])
    status = data["status"]

    # The strategic target is permanent unless manually changed.
    strategic_pct = (
        strategic_gbp / NOTIONAL_PORTFOLIO_GBP
        if NOTIONAL_PORTFOLIO_GBP > 0
        else 0
    )

    # The trend-following overlay controls whether the strategic
    # allocation is currently active.
    if status == "EXIT":
        active_target_gbp = 0.0
    else:
        active_target_gbp = strategic_gbp

    active_target_pct = (
        active_target_gbp / NOTIONAL_PORTFOLIO_GBP
        if NOTIONAL_PORTFOLIO_GBP > 0
        else 0
    )

    trend_excluded_gbp = strategic_gbp - active_target_gbp

    trend_excluded_pct = (
        trend_excluded_gbp / NOTIONAL_PORTFOLIO_GBP
        if NOTIONAL_PORTFOLIO_GBP > 0
        else 0
    )

    # Rebalancing bands apply to ACTIVE target exposure.
    if active_target_gbp == 0:
        lower_bound_gbp = 0.0
        upper_bound_gbp = 0.0
    else:
        lower_bound_gbp = active_target_gbp * (1 - LOWER_BAND_PCT)
        upper_bound_gbp = active_target_gbp * (1 + UPPER_BAND_PCT)

    lower_bound_pct = (
        lower_bound_gbp / NOTIONAL_PORTFOLIO_GBP
        if NOTIONAL_PORTFOLIO_GBP > 0
        else 0
    )

    upper_bound_pct = (
        upper_bound_gbp / NOTIONAL_PORTFOLIO_GBP
        if NOTIONAL_PORTFOLIO_GBP > 0
        else 0
    )

    rows.append({
        "Asset": asset,
        "Trend Signal": status,

        "Strategic GBP": strategic_gbp,
        "Strategic %": strategic_pct,

        "Active Target GBP": active_target_gbp,
        "Active Target %": active_target_pct,

        "Trend-Excluded GBP": trend_excluded_gbp,
        "Trend-Excluded %": trend_excluded_pct,

        "Lower Bound GBP": lower_bound_gbp,
        "Lower Bound %": lower_bound_pct,

        "Upper Bound GBP": upper_bound_gbp,
        "Upper Bound %": upper_bound_pct,
    })


allocation_df = pd.DataFrame(rows).set_index("Asset")


# ============================================================
# ADD CURRENT PORTFOLIO
# ============================================================

allocation_df["Current GBP"] = [
    float(CURRENT_PORTFOLIO_GBP[asset])
    for asset in allocation_df.index
]

allocation_df["Current %"] = (
    allocation_df["Current GBP"]
    / NOTIONAL_PORTFOLIO_GBP
)


# ============================================================
# CALCULATE DELTAS
# ============================================================
#
# Strategic Delta:
#   Distance from the long-run strategic allocation.
#
# Active Delta:
#   Distance from the exposure currently authorised by the
#   trend-following overlay.
#
# ============================================================

allocation_df["Delta to Strategic GBP"] = (
    allocation_df["Strategic GBP"]
    - allocation_df["Current GBP"]
)

allocation_df["Delta to Strategic %"] = (
    allocation_df["Strategic %"]
    - allocation_df["Current %"]
)

allocation_df["Delta to Active Target GBP"] = (
    allocation_df["Active Target GBP"]
    - allocation_df["Current GBP"]
)

allocation_df["Delta to Active Target %"] = (
    allocation_df["Active Target %"]
    - allocation_df["Current %"]
)


# ============================================================
# CLASSIFY POSITION RELATIVE TO ACTIVE TARGET BAND
# ============================================================

def classify_position(row):

    current = row["Current GBP"]
    active_target = row["Active Target GBP"]
    lower = row["Lower Bound GBP"]
    upper = row["Upper Bound GBP"]
    status = row["Trend Signal"]

    if status == "EXIT":

        if current > 0:
            return "EXIT SIGNAL — EXPOSURE REMAINS"

        return "EXIT — NO EXPOSURE"

    if current > upper:
        return "ABOVE UPPER BAND"

    if current < lower:
        return "BELOW LOWER BAND"

    return "WITHIN BAND"


allocation_df["Position State"] = allocation_df.apply(
    classify_position,
    axis=1,
)


# ============================================================
# EXECUTION ENGINE
# ============================================================
#
# Recommended Trade GBP:
#
# Positive = BUY
# Negative = SELL
# Zero     = HOLD / NO ACTION
#
# ============================================================

def calculate_trade(row):

    current = row["Current GBP"]
    active_target = row["Active Target GBP"]
    upper = row["Upper Bound GBP"]
    status = row["Trend Signal"]

    # --------------------------------------------------------
    # MONTHLY REBALANCE
    # --------------------------------------------------------
    #
    # SELL ONLY.
    #
    # If current exposure is above the upper ACTIVE target band,
    # sell down TO the upper bound.
    #
    # If the trend signal is EXIT, active target and upper band
    # are both zero, so any remaining exposure is sold.
    #
    # No purchases are generated in this mode.
    #
    # --------------------------------------------------------

    if RUN_MODE == "MONTHLY_REBALANCE":

        if current > upper:
            return upper - current

        return 0.0

    # --------------------------------------------------------
    # INTRAMONTH ACCUMULATION
    # --------------------------------------------------------
    #
    # BUY ONLY.
    #
    # Only assets explicitly marked ACCUMULATION can receive
    # additional capital.
    #
    # The buy closes a configurable proportion of the gap
    # between CURRENT exposure and ACTIVE target exposure.
    #
    # HOLD underweights do not generate purchases.
    #
    # EXIT positions are flagged, but are not sold in this mode.
    #
    # --------------------------------------------------------

    elif RUN_MODE == "INTRAMONTH_ACCUMULATION":

        if (
            status == "ACCUMULATION"
            and current < active_target
        ):

            full_gap = active_target - current

            monthly_purchase = (
                full_gap
                * ACCUMULATION_FILL_FRACTION
            )

            return monthly_purchase

        return 0.0


allocation_df["Recommended Trade GBP"] = allocation_df.apply(
    calculate_trade,
    axis=1,
)


allocation_df["Recommended Trade %"] = (
    allocation_df["Recommended Trade GBP"]
    / NOTIONAL_PORTFOLIO_GBP
)


# ============================================================
# ACTION LABEL
# ============================================================

def trade_action(value):

    if value > 0:
        return "BUY"

    if value < 0:
        return "SELL"

    return "HOLD"


allocation_df["Action"] = (
    allocation_df["Recommended Trade GBP"]
    .apply(trade_action)
)


# ============================================================
# POST-TRADE EXPOSURE
# ============================================================

allocation_df["Post-Trade GBP"] = (
    allocation_df["Current GBP"]
    + allocation_df["Recommended Trade GBP"]
)

allocation_df["Post-Trade %"] = (
    allocation_df["Post-Trade GBP"]
    / NOTIONAL_PORTFOLIO_GBP
)


# ============================================================
# DCA CALCULATIONS
# ============================================================

allocation_df["DCA Per Instalment GBP"] = 0.0

if RUN_MODE == "INTRAMONTH_ACCUMULATION":

    buy_mask = (
        allocation_df["Recommended Trade GBP"] > 0
    )

    allocation_df.loc[
        buy_mask,
        "DCA Per Instalment GBP"
    ] = (
        allocation_df.loc[
            buy_mask,
            "Recommended Trade GBP"
        ]
        / DCA_INSTALLMENTS
    )


# ============================================================
# NOTES / TREND-OVERLAY WARNINGS
# ============================================================

allocation_df["Note"] = ""

for asset in allocation_df.index:

    status = allocation_df.loc[asset, "Trend Signal"]
    current = allocation_df.loc[asset, "Current GBP"]

    if (
        RUN_MODE == "INTRAMONTH_ACCUMULATION"
        and status == "EXIT"
        and current > 0
    ):
        allocation_df.loc[
            asset,
            "Note"
        ] = (
            "EXIT signal: active target = £0; sale deferred "
            "until MONTHLY_REBALANCE mode"
        )

    elif (
        status == "HOLD"
        and current < allocation_df.loc[asset, "Lower Bound GBP"]
    ):
        allocation_df.loc[
            asset,
            "Note"
        ] = (
            "Below target, but HOLD signal does not authorise "
            "intra-month accumulation"
        )


# ============================================================
# PORTFOLIO-LEVEL STATISTICS
# ============================================================

strategic_total_gbp = allocation_df["Strategic GBP"].sum()
active_target_gbp = allocation_df["Active Target GBP"].sum()
trend_excluded_gbp = allocation_df["Trend-Excluded GBP"].sum()

current_exposure_gbp = allocation_df["Current GBP"].sum()

strategic_exposure_multiplier = (
    strategic_total_gbp / CASH_CAPITAL_GBP
    if CASH_CAPITAL_GBP > 0
    else np.nan
)

active_exposure_multiplier = (
    active_target_gbp / CASH_CAPITAL_GBP
    if CASH_CAPITAL_GBP > 0
    else np.nan
)

current_exposure_multiplier = (
    current_exposure_gbp / CASH_CAPITAL_GBP
    if CASH_CAPITAL_GBP > 0
    else np.nan
)

recommended_net_trade_gbp = (
    allocation_df["Recommended Trade GBP"].sum()
)

recommended_buys_gbp = (
    allocation_df.loc[
        allocation_df["Recommended Trade GBP"] > 0,
        "Recommended Trade GBP"
    ].sum()
)

recommended_sells_gbp = -(
    allocation_df.loc[
        allocation_df["Recommended Trade GBP"] < 0,
        "Recommended Trade GBP"
    ].sum()
)

post_trade_exposure_gbp = (
    allocation_df["Post-Trade GBP"].sum()
)

post_trade_exposure_multiplier = (
    post_trade_exposure_gbp / CASH_CAPITAL_GBP
    if CASH_CAPITAL_GBP > 0
    else np.nan
)


# ============================================================
# DISPLAY HELPERS
# ============================================================

def gbp(x):
    return f"£{x:,.2f}"


def pct(x):
    return f"{x:.2%}"


def multiple(x):
    return f"{x:.2f}x"


# ============================================================
# PORTFOLIO SUMMARY
# ============================================================

summary_df = pd.DataFrame({

    "Metric": [
        "Cash Capital",
        "Configured Exposure Multiplier",
        "Notional Strategic Portfolio",
        "Strategic Allocation Total",
        "Strategic Exposure Multiplier",
        "Active Target Exposure",
        "Active Exposure Multiplier",
        "Trend-Excluded Exposure",
        "Current Exposure",
        "Current Exposure Multiplier",
        "Recommended Buys",
        "Recommended Sells",
        "Net Recommended Trade",
        "Post-Trade Exposure",
        "Post-Trade Exposure Multiplier",
    ],

    "Value": [
        gbp(CASH_CAPITAL_GBP),
        multiple(EXPOSURE_MULTIPLIER),
        gbp(NOTIONAL_PORTFOLIO_GBP),
        gbp(strategic_total_gbp),
        multiple(strategic_exposure_multiplier),
        gbp(active_target_gbp),
        multiple(active_exposure_multiplier),
        gbp(trend_excluded_gbp),
        gbp(current_exposure_gbp),
        multiple(current_exposure_multiplier),
        gbp(recommended_buys_gbp),
        gbp(recommended_sells_gbp),
        gbp(recommended_net_trade_gbp),
        gbp(post_trade_exposure_gbp),
        multiple(post_trade_exposure_multiplier),
    ]
})


# ============================================================
# STRATEGIC + ACTIVE TARGET OUTPUT
# ============================================================

strategic_output = allocation_df[[
    "Trend Signal",
    "Strategic GBP",
    "Strategic %",
    "Active Target GBP",
    "Active Target %",
    "Trend-Excluded GBP",
    "Trend-Excluded %",
    "Lower Bound GBP",
    "Lower Bound %",
    "Upper Bound GBP",
    "Upper Bound %",
]].copy()


# ============================================================
# CURRENT POSITION / DELTA OUTPUT
# ============================================================

position_output = allocation_df[[
    "Trend Signal",
    "Strategic GBP",
    "Strategic %",
    "Active Target GBP",
    "Active Target %",
    "Current GBP",
    "Current %",
    "Delta to Strategic GBP",
    "Delta to Strategic %",
    "Delta to Active Target GBP",
    "Delta to Active Target %",
    "Position State",
]].copy()


# ============================================================
# EXECUTION OUTPUT
# ============================================================

execution_output = allocation_df[[
    "Trend Signal",
    "Position State",
    "Action",
    "Recommended Trade GBP",
    "Recommended Trade %",
    "DCA Per Instalment GBP",
    "Current GBP",
    "Post-Trade GBP",
    "Post-Trade %",
    "Note",
]].copy()


# Only show actual trades in separate action table.

trades_only = execution_output[
    execution_output["Action"] != "HOLD"
].copy()


# ============================================================
# FORMAT TABLES FOR DISPLAY
# ============================================================

GBP_COLUMNS = [
    "Strategic GBP",
    "Active Target GBP",
    "Trend-Excluded GBP",
    "Lower Bound GBP",
    "Upper Bound GBP",
    "Current GBP",
    "Delta to Strategic GBP",
    "Delta to Active Target GBP",
    "Recommended Trade GBP",
    "DCA Per Instalment GBP",
    "Post-Trade GBP",
]

PCT_COLUMNS = [
    "Strategic %",
    "Active Target %",
    "Trend-Excluded %",
    "Lower Bound %",
    "Upper Bound %",
    "Current %",
    "Delta to Strategic %",
    "Delta to Active Target %",
    "Recommended Trade %",
    "Post-Trade %",
]


def format_table(df):

    result = df.copy()

    for col in GBP_COLUMNS:
        if col in result.columns:
            result[col] = result[col].map(
                lambda x: f"£{x:,.2f}"
            )

    for col in PCT_COLUMNS:
        if col in result.columns:
            result[col] = result[col].map(
                lambda x: f"{x:.2%}"
            )

    return result


# ============================================================
# VALIDATION WARNINGS
# ============================================================

print("=" * 80)
print("STRATEGIC ASSET ALLOCATION & TREND-FOLLOWING OVERLAY FRAMEWORK")
print("=" * 80)

print(f"\nRUN MODE: {RUN_MODE}")

print(
    f"Cash Capital:                  "
    f"{gbp(CASH_CAPITAL_GBP)}"
)

print(
    f"Configured Exposure Multiplier:"
    f" {multiple(EXPOSURE_MULTIPLIER)}"
)

print(
    f"Notional Strategic Portfolio:  "
    f"{gbp(NOTIONAL_PORTFOLIO_GBP)}"
)


strategic_difference = (
    strategic_total_gbp
    - NOTIONAL_PORTFOLIO_GBP
)

if abs(strategic_difference) > 0.01:

    print("\n⚠️ STRATEGIC ALLOCATION WARNING")

    print(
        f"Strategic targets total "
        f"{gbp(strategic_total_gbp)}, "
        f"but the configured notional portfolio is "
        f"{gbp(NOTIONAL_PORTFOLIO_GBP)}."
    )

    print(
        f"Difference: "
        f"{gbp(strategic_difference)}"
    )

else:

    print(
        "\n✓ Strategic allocations equal the configured "
        "notional portfolio."
    )


if trend_excluded_gbp > 0.01:

    print(
        f"\nActive target exposure:       "
        f"{gbp(active_target_gbp)} "
        f"({multiple(active_exposure_multiplier)})"
    )

    print(
        f"Trend-excluded exposure:       "
        f"{gbp(trend_excluded_gbp)}"
    )

    print(
        "This exposure remains part of the long-run strategic "
        "allocation but is temporarily inactive because one or "
        "more asset classes currently have an EXIT signal."
    )


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n\n1. PORTFOLIO SUMMARY")
print("-" * 80)

display(summary_df)


print("\n\n2. STRATEGIC ALLOCATION & ACTIVE TREND TARGETS")
print("-" * 80)

display(
    format_table(strategic_output)
)


print("\n\n3. CURRENT PORTFOLIO VS STRATEGIC / ACTIVE TARGETS")
print("-" * 80)

display(
    format_table(position_output)
)


print("\n\n4. RECOMMENDED ACTIONS")
print("-" * 80)

display(
    format_table(execution_output)
)


print("\n\n5. TRADES TO EXECUTE")
print("-" * 80)

if len(trades_only) == 0:

    print("No trades required under the current execution rules.")

else:

    display(
        format_table(trades_only)
    )


# ============================================================
# PLAIN-ENGLISH EXECUTION SUMMARY
# ============================================================

print("\n\nEXECUTION SUMMARY")
print("=" * 80)


if RUN_MODE == "MONTHLY_REBALANCE":

    print(
        "\nMONTHLY REBALANCE MODE is SELL-ONLY."
    )

    print(
        "Positions are trimmed only if they exceed their "
        "upper ACTIVE target band."
    )

    print(
        "Assets carrying an EXIT signal have an active target "
        "of £0, so remaining exposure is sold in this mode.\n"
    )

elif RUN_MODE == "INTRAMONTH_ACCUMULATION":

    print(
        "\nINTRAMONTH ACCUMULATION MODE is BUY-ONLY."
    )

    print(
        "Only assets carrying an ACCUMULATION signal from the "
        "separate trend-following model are eligible for buys."
    )

    print(
        "HOLD assets retain their strategic targets but do not "
        "receive new purchases in this mode."
    )

    print(
        "EXIT positions are flagged but are not sold until "
        "MONTHLY_REBALANCE mode is run.\n"
    )


for asset, row in allocation_df.iterrows():

    trade = row["Recommended Trade GBP"]

    if trade > 0:

        print(
            f"BUY  {asset:<20} "
            f"{gbp(trade):>15}"
            f"  |  "
            f"{pct(row['Recommended Trade %']):>8}"
            f" of strategic portfolio"
        )

        if DCA_INSTALLMENTS > 1:

            print(
                f"     ↳ {DCA_INSTALLMENTS} DCA purchases "
                f"of approximately "
                f"{gbp(row['DCA Per Instalment GBP'])}"
            )

    elif trade < 0:

        print(
            f"SELL {asset:<20} "
            f"{gbp(abs(trade)):>15}"
            f"  |  "
            f"{pct(abs(row['Recommended Trade %'])):>8}"
            f" of strategic portfolio"
        )


if np.isclose(
    allocation_df["Recommended Trade GBP"].abs().sum(),
    0
):

    print(
        "No trades are required under the current "
        "execution rules."
    )


STRATEGIC ASSET ALLOCATION & TREND-FOLLOWING OVERLAY FRAMEWORK

RUN MODE: INTRAMONTH_ACCUMULATION
Cash Capital:                  £100,000.00
Configured Exposure Multiplier: 1.20x
Notional Strategic Portfolio:  £120,000.00

✓ Strategic allocations equal the configured notional portfolio.

Active target exposure:       £105,000.00 (1.05x)
Trend-excluded exposure:       £15,000.00
This exposure remains part of the long-run strategic allocation but is temporarily inactive because one or more asset classes currently have an EXIT signal.


1. PORTFOLIO SUMMARY
--------------------------------------------------------------------------------


,Metric,Value
0,Cash Capital,"£100,000.00"
1,Configured Exposure Multiplier,1.20x
2,Notional Strategic Portfolio,"£120,000.00"
3,Strategic Allocation Total,"£120,000.00"
4,Strategic Exposure Multiplier,1.20x
5,Active Target Exposure,"£105,000.00"
6,Active Exposure Multiplier,1.05x
7,Trend-Excluded Exposure,"£15,000.00"
8,Current Exposure,"£120,000.00"
9,Current Exposure Multiplier,1.20x




2. STRATEGIC ALLOCATION & ACTIVE TREND TARGETS
--------------------------------------------------------------------------------


,Trend Signal,Strategic GBP,Strategic %,Active Target GBP,Active Target %,Trend-Excluded GBP,Trend-Excluded %,Lower Bound GBP,Lower Bound %,Upper Bound GBP,Upper Bound %
Asset,,,,,,,,,,,
EQUITIES_US,HOLD,"£48,000.00",40.00%,"£48,000.00",40.00%,£0.00,0.00%,"£43,200.00",36.00%,"£52,800.00",44.00%
EQUITIES_DM_EX_US,HOLD,"£15,000.00",12.50%,"£15,000.00",12.50%,£0.00,0.00%,"£13,500.00",11.25%,"£16,500.00",13.75%
EQUITIES_EM,EXIT,"£15,000.00",12.50%,£0.00,0.00%,"£15,000.00",12.50%,£0.00,0.00%,£0.00,0.00%
BITCOIN,ACCUMULATION,"£12,000.00",10.00%,"£12,000.00",10.00%,£0.00,0.00%,"£10,800.00",9.00%,"£13,200.00",11.00%
BONDS,HOLD,"£12,000.00",10.00%,"£12,000.00",10.00%,£0.00,0.00%,"£10,800.00",9.00%,"£13,200.00",11.00%
DIVERSIFIERS,HOLD,"£12,000.00",10.00%,"£12,000.00",10.00%,£0.00,0.00%,"£10,800.00",9.00%,"£13,200.00",11.00%
CASH,HOLD,"£6,000.00",5.00%,"£6,000.00",5.00%,£0.00,0.00%,"£5,400.00",4.50%,"£6,600.00",5.50%




3. CURRENT PORTFOLIO VS STRATEGIC / ACTIVE TARGETS
--------------------------------------------------------------------------------


...



4. RECOMMENDED ACTIONS
--------------------------------------------------------------------------------


,Trend Signal,Position State,Action,Recommended Trade GBP,Recommended Trade %,DCA Per Instalment GBP,Current GBP,Post-Trade GBP,Post-Trade %,Note
Asset,,,,,,,,,,
EQUITIES_US,HOLD,ABOVE UPPER BAND,HOLD,£0.00,0.00%,£0.00,"£54,000.00","£54,000.00",45.00%,
EQUITIES_DM_EX_US,HOLD,WITHIN BAND,HOLD,£0.00,0.00%,£0.00,"£16,000.00","£16,000.00",13.33%,
EQUITIES_EM,EXIT,EXIT SIGNAL — EXPOSURE REMAINS,HOLD,£0.00,0.00%,£0.00,"£11,000.00","£11,000.00",9.17%,EXIT signal: active target = £0; sale deferred...
BITCOIN,ACCUMULATION,BELOW LOWER BAND,BUY,"£3,000.00",2.50%,£750.00,"£9,000.00","£12,000.00",10.00%,
BONDS,HOLD,BELOW LOWER BAND,HOLD,£0.00,0.00%,£0.00,"£10,000.00","£10,000.00",8.33%,"Below target, but HOLD signal does not authori..."
DIVERSIFIERS,HOLD,ABOVE UPPER BAND,HOLD,£0.00,0.00%,£0.00,"£15,000.00","£15,000.00",12.50%,
CASH,HOLD,BELOW LOWER BAND,HOLD,£0.00,0.00%,£0.00,"£5,000.00","£5,000.00",4.17%,"Below target, but HOLD signal does not authori..."




5. TRADES TO EXECUTE
--------------------------------------------------------------------------------


,Trend Signal,Position State,Action,Recommended Trade GBP,Recommended Trade %,DCA Per Instalment GBP,Current GBP,Post-Trade GBP,Post-Trade %,Note
Asset,,,,,,,,,,
BITCOIN,ACCUMULATION,BELOW LOWER BAND,BUY,"£3,000.00",2.50%,£750.00,"£9,000.00","£12,000.00",10.00%,




EXECUTION SUMMARY

INTRAMONTH ACCUMULATION MODE is BUY-ONLY.
Only assets carrying an ACCUMULATION signal from the separate trend-following model are eligible for buys.
HOLD assets retain their strategic targets but do not receive new purchases in this mode.
EXIT positions are flagged but are not sold until MONTHLY_REBALANCE mode is run.

BUY  BITCOIN                    £3,000.00  |     2.50% of strategic portfolio
     ↳ 4 DCA purchases of approximately £750.00
